In [1]:
"""本地测试脚本 - 不依赖 FastAPI"""
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI

# 添加项目根目录到 Python 路径
# 获取项目根目录（tests 目录的父目录）
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 从 rag 模块统一导入
from rag import (
    build_vector_db,
    hybrid_search,
    dense_search,
    sparse_search,
    two_stage_retrieval,
    get_doc_count,
    get_all_docs,
    resolve_placeholders,
    router,
    rewrite,
    rewrite_for_retrieval,
)

load_dotenv()

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# ============ 配置 ============
REPORT_PATH =  "D:/github/rag_project/rag_财报/财报" 
API_KEY = os.getenv('DEEPSEEK_API_KEY')
API_BASE = os.getenv('DEEPSEEK_API_BASE')
MODEL = os.getenv('DEEPSEEK_MODEL')

client = OpenAI(api_key=API_KEY, base_url=API_BASE)

model = None

history = []  # 对话历史

def init_system():
    """初始化系统"""
    global model, history
    
    # 初始化向量数据库
    model = build_vector_db(REPORT_PATH, rebuild=False)
    
    # 系统提示词
    system_prompt= """
# 角色设定
你是一个专业、客观、严谨的财务分析助手。你的核心任务是基于提供的【检索参考】提取关键信息，准确回答用户的财务与公司基本面问题。

# 处理流程
请严格按照以下步骤处理用户的输入（Query）：

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
【步骤1】意图识别与分类（核心判断）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
首先分析用户的输入，将其归入以下两类之一，并执行对应的行动指引：

**类型A：非财务/非检索类问题（闲聊、常识、无关提问）**
*   **特征**：日常问候（你好、谢谢）、百科常识（今天天气、某位明星是谁）、要求执行非财务任务（写首诗、讲个笑话）。
*   **行动指引**：
    1. 完全忽略并丢弃下方的【检索参考】。
    2. 保持礼貌，简短回应。
    3. 拒绝回答与财务/公司经营无关的具体知识，并主动将话题引导回财务分析上。
*   **示例**：
    *   用户："你好"
    *   助手："您好！我是财务分析助手，请问有什么财报数据或公司信息需要我为您查询吗？"
    *   用户："特朗普回美国了吗？"
    *   助手："抱歉，作为财务分析助手，我主要关注公司财报、经营数据和商业资讯。关于日常新闻建议您查阅相关媒体。请问有特定公司的财务状况需要我帮您分析吗？"


类型B：事实检索类问题（询问财报中明确记录的数据、公司基本面事实）
→ 应对策略：直接从【检索参考】中提取关键数据或事实回答，不需要分析过程。如果没有相关信息，回答"根据提供的财报内容，暂无相关数据。"

类型C：分析测算与假设推演类问题
→ 应对策略：允许进行逻辑推理与数学计算，进入【步骤3-核心计算规则】。

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
【步骤2】基于【检索参考】提取与回答（仅限类型B）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
当判断为类型B时，你必须且只能根据下文提供的【检索参考】进行作答。

*   **规则 1（精准提取）**：如果参考中有相关信息，直接给出客观的关键数据或事实，**不需要**输出你的推理过程。
*   **规则 2（严格防幻觉）**：如果参考中没有提到相关信息，回复："根据提供的财报内容，我无法回答该问题。"，也要告诉用户回答不了的原因. **绝不允许**动用自身内部知识库编造或推测数据。

*   **示例**：
    *   用户："盛和资源2019年的研发投入是多少？"
    *   检索参考：【盛和资源2019年报】研发投入1.2亿元...
    *   助手："盛和资源2019年研发投入为1.2亿元。"

    
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
【步骤3】核心计算规则（针对类型C）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
当回答"分析测算类"问题时，请严格遵守以下推导规则, 不需要输出有关核心计算规则的文字, 你了解就行：

1. 提取变量：仔细阅读【检索参考】，提取出解题所需的所有已知财务数据，并在回答开头明确列出。
2. 严禁编造变量：如果推导公式中必须用到的某个关键数据在【检索参考】中**完全找不到**，你必须停止计算，并告知用户："要计算此问题需要用到[XX数据]，但检索结果中未提供，因此无法完成测算。"绝不允许自己凭空捏造数据填补。
3. 展示精简的推导过程：简单展示核心的计算公式或逻辑链条, 无需展示所有计算步骤, 展示大致的核心步骤即可。
4. 财务逻辑修正：如果用户的假设本身存在财务逻辑漏洞（如混淆了毛利润和净利润、忽略了期间费用与所得税），你作为专业助手需要委婉指出，并基于更严谨的财务逻辑给出你的测算。
5. 免责声明：在推演类回答的最后，必须加上一句："（注：以上推演基于假设条件测算，不构成投资建议或真实的财务预测。）"

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
【步骤4】输出前最终核对（内部检查，不输出此部分）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
在生成最终回复前，请自行核对以下三点：
1. 若为类型B，所有数据是否 100% 来源于【检索参考】？
2. 是否存在任何形式的主观推测、股票推荐或投资建议？（必须避免）
3. 语言风格是否专业、清晰、精炼？

"""


    # 清空历史并添加系统提示词
    history = [{"role": "system", "content": system_prompt}]
    
    # 显示状态
    doc_count = get_doc_count()
    print(f"[2/2] 初始化完成！")
    print(f"   - 当前知识库文本块数: {doc_count}")
    print(f"   - 财报文件: {REPORT_PATH}")
    print("=" * 60)


def test_search(query: str, mode: str = "hybrid", top_k: int = 5, show_sources: bool = True, source_filter: str = None):
    """
    测试检索功能

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 返回结果数
        show_sources: 是否显示检索到的原始片段
        source_filter: 指定检索的源文件
    """
    print(f"\n{'='*60}")
    print(f" 查询问题: {query}")
    print(f" 检索模式: {mode}")
    print(f" 返回数量: {top_k}")
    print("=" * 60)

    # 根据mode执行不同的检索
    if mode == "dense":
        results = dense_search(query, top_k=top_k, source_filter=source_filter)
        print(" 向量检索 (BGE向量)")
    elif mode == "sparse":
        results = sparse_search(query, top_k=top_k, source_filter=source_filter)
        print(" 关键词检索 (BM25)")
    else:  # hybrid
        results = hybrid_search(query, top_k=top_k, source_filter=source_filter)
        # 同时获取密集检索结果用于显示得分
        dense_results = dense_search(query, top_k=10, source_filter=source_filter)
        print(" 混合检索 (BGE向量 + BM25 + RRF)")

    # 显示检索结果
    print(f"\n 检索到 {len(results)} 条相关片段:")
    print("-" * 60)

    for i, (doc_id, chunk_text, score, source_file) in enumerate(results, 1):
            if mode == "dense":
                # 将距离转换为相似度：similarity = 1 / (1 + distance)
                similarity = 1 / (1 + score)
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
            elif mode == "sparse":
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  BM25得分: {score:.4f}")
            else:  # hybrid
                d_score = {d_id: s for d_id, _, s, _ in dense_results}.get(doc_id, float('inf'))
                similarity = 1 / (1 + d_score) if d_score != float('inf') else 0
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  RRF得分: {score:.4f} | 向量相似度: {similarity:.4f}")
            
            if show_sources:
                print(f"  内容: {chunk_text}")
    return results


def test_generation(query: str, mode: str = "hybrid", top_k: int = 5):
    """
    测试完整流程（检索 + 生成）

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 检索返回数量
    """
    global history, last_retrieved_chunks
    
    # 保存原始query
    original_query = query
    
    # --- 1. Query Rewriting（处理代词、省略）---
    query = rewrite(query, history)
    
    # --- 2. LLM 智能路由 ---
    target_docs = router(query)
    
    # --- 3. 第二步改写 ---
    query = rewrite_for_retrieval(query)
    print(f'改写后用于检索的query:{query}')

    # --- 4. 检索逻辑：支持跨文档分别检索 ---
    all_results = []
    
    # 判断是否为全局查询（包含"全部"或文档列表长度等于全部文档数）
    all_docs_list = get_all_docs()
    is_global_query = "全部" in target_docs or len(target_docs) == len(all_docs_list)
    
    if is_global_query:
        # 全局查询：使用两阶段检索
        if mode == "hybrid":
            print(f"\n[两阶段检索] 全局查询，使用粗排+Rerank精排...")
            results = two_stage_retrieval(query, top_k=top_k)
            # 显示结果（包含片段内容）
            print(f"\n 检索到 {len(results)} 条相关片段:")
            print("-" * 60)
            for i, (doc_id, chunk_text, score, source_file) in enumerate(results, 1):
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  Rerank融合得分: {score:.4f}")
                print(f"  内容: {chunk_text}")
            all_results = results
        else:
            # dense/sparse 模式保持原逻辑
            results = test_search(query, mode=mode, top_k=top_k, source_filter=None)
            all_results = results
    elif len(target_docs) == 1:
        # 单个文档 - 保持原逻辑
        results = test_search(query, mode=mode, top_k=top_k, source_filter=target_docs[0])
        all_results = results
    else:
        # 多个文档：分别检索每个文档，然后合并
        print(f" 跨文档查询，分别检索 {len(target_docs)} 个文档...")
        for doc in target_docs:
            print(f"\n  正在检索: {doc}")
            results = test_search(query, mode=mode, top_k=top_k, source_filter=doc)
            all_results.extend(results)
        
        # 按分数重新排序并取top_k
        if mode == "hybrid":
            all_results.sort(key=lambda x: x[2], reverse=True)
        elif mode == "dense":
            all_results.sort(key=lambda x: x[2])  # 距离越小越好
        elif mode == "sparse":
            all_results.sort(key=lambda x: x[2], reverse=True)
        
        all_results = all_results[:top_k]
        print(f"\n 合并后取 top-{len(all_results)} 结果")

    if not all_results:
        print("\n 无法生成回答：未找到相关内容")
        return

    # --- 5. 解析占位符 ---
    print("\n 正在解析表格占位符...")
    all_results = resolve_placeholders(all_results)
    
    retrieved_contexts = []
    for _, text, _, source in all_results:
        # 去掉 .md 后缀
        clean_source = source.replace('.md', '')
        # 拼接成带有来源标识的区块
        context_block = f"【来源文件：{clean_source}】\n{text}"
        retrieved_contexts.append(context_block)
        
    context_text = "\n\n---\n\n".join(retrieved_contexts)

    # 删除历史中旧的财报片段
    history[:] = [msg for msg in history if not (msg.get('role') == 'system' and '【本轮财报片段】' in msg.get('content', ''))]
    
    # 添加新的财报片段和用户问题（使用原始query）
    history.append({"role": "system", "content": f"【本轮财报片段】\n{context_text}"})
    history.append({"role": "user", "content": original_query})
    
    # 直接用 history 调用 LLM
    response = client.chat.completions.create(
        model=MODEL,    
        extra_body={"thinking": {"type": "disabled"}},
        messages=history,
        temperature=0.1
    )
    answer = response.choices[0].message.content
    
    history.append({"role": "assistant", "content": answer})
    
    print('\nAI 回答')
    print(f'AI回答: {answer}')

In [14]:
init_system()
query = '合诚股份子公司大连市政设计院、厦门合诚检测、厦门合诚工程、厦门合诚设计院四家子公司2019年的营业收入总和是多少？'
results = test_generation(query, top_k=10)

数据库中已有 3273 条记录
[2/2] 初始化完成！
   - 当前知识库文本块数: 3273
   - 财报文件: D:/github/rag_project/rag_财报/财报
🤖 LLM返回: ['合诚工程咨询集团股份有限公司2019 年年度报告.md']
✅ 匹配到文档: ['合诚工程咨询集团股份有限公司2019 年年度报告.md']
 检索优化rewrite: 合诚股份子公司大连市政设计院、厦门合诚检测、厦门合诚工程、厦门合诚设计院四家子公司2019年的营业收入总和是多少？ → 大连市政设计院 厦门合诚检测 厦门合诚工程 厦门合诚设计院 2019年 营业收入
改写后用于检索的query:大连市政设计院 厦门合诚检测 厦门合诚工程 厦门合诚设计院 2019年 营业收入

 查询问题: 大连市政设计院 厦门合诚检测 厦门合诚工程 厦门合诚设计院 2019年 营业收入
 检索模式: hybrid
 返回数量: 10
 混合检索 (BGE向量 + BM25 + RRF)

 检索到 10 条相关片段:
------------------------------------------------------------

[片段 1] ID: 1177 | 来源: 合诚工程咨询集团股份有限公司2019 年年度报告.md
  RRF得分: 0.0309 | 向量相似度: 0.8246
  内容: <html><body><table><tr><td>厦门合诚工程设计院 有限公司</td><td>9,375,230.92</td><td>815,622.18</td><td></td><td>10,190,853.10</td></tr><tr><td>厦门合诚工程技术有 限公司</td><td>16,252,251.14</td><td>1,230,137.54</td><td></td><td>17,482,388.68</td></tr><tr><td>厦门合智新材料科技 有限公司</td><td>10,742,074.75</td><td>728,970.39</td><td></td><td>11,471,045.14</td></tr><tr><td>湖南合友工程检测有 限公司</td><td>3,106,556.06</td><td>608,600

In [8]:
def compare_retrieval_methods(query: str, top_k: int = 10):
    """
    对比两阶段检索和普通混合检索的召回结果
    
    Args:
        query: 查询问题
        top_k: 返回结果数
    """
    print(f"\n{'='*80}")
    print(f" 对比测试：两阶段检索 vs 普通混合检索")
    print(f" 查询问题: {query}")
    print(f" 返回数量: {top_k}")
    print('='*80)
    
    # ============ 方法1：普通混合检索 ============
    print(f"\n{'─'*80}")
    print(f" 【方法1】普通混合检索 (Hybrid + RRF)")
    print(f"{'─'*80}")
    
    hybrid_results = hybrid_search(query, top_k=top_k)
    
    # 统计各公司片段数量
    hybrid_companies = {}
    for doc_id, chunk_text, score, source_file in hybrid_results:
        company = source_file.replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
        if company not in hybrid_companies:
            hybrid_companies[company] = []
        hybrid_companies[company].append(doc_id)
    
    print(f"\n 检索到 {len(hybrid_results)} 条片段")
    print(f" 覆盖公司数: {len(hybrid_companies)}")
    print(f" 公司分布: {dict(sorted(hybrid_companies.items(), key=lambda x: len(x[1]), reverse=True))}")
    
    print(f"\n 片段详情:")
    for i, (doc_id, chunk_text, score, source_file) in enumerate(hybrid_results, 1):
        company = source_file.replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
        # 截取内容前100字符
        content_preview = chunk_text[:100].replace('\n', ' ') + '...' if len(chunk_text) > 100 else chunk_text
        print(f"   [{i}] {company} | ID:{doc_id} | 得分:{score:.4f}")
        print(f"       {content_preview}")
    
    # ============ 方法2：两阶段检索 ============
    print(f"\n{'─'*80}")
    print(f" 【方法2】两阶段检索 (粗排{top_k*3} + Rerank精排)")
    print(f"{'─'*80}")
    
    two_stage_results = two_stage_retrieval(query, top_k=top_k)
    
    # 统计各公司片段数量
    two_stage_companies = {}
    for doc_id, chunk_text, score, source_file in two_stage_results:
        company = source_file.replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
        if company not in two_stage_companies:
            two_stage_companies[company] = []
        two_stage_companies[company].append(doc_id)
    
    print(f"\n 检索到 {len(two_stage_results)} 条片段")
    print(f" 覆盖公司数: {len(two_stage_companies)}")
    print(f" 公司分布: {dict(sorted(two_stage_companies.items(), key=lambda x: len(x[1]), reverse=True))}")
    
    print(f"\n 片段详情:")
    for i, (doc_id, chunk_text, score, source_file) in enumerate(two_stage_results, 1):
        company = source_file.replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
        content_preview = chunk_text[:100].replace('\n', ' ') + '...' if len(chunk_text) > 100 else chunk_text
        print(f"   [{i}] {company} | ID:{doc_id} | 得分:{score:.4f}")
        print(f"       {content_preview}")
    
    # ============ 对比分析 ============
    print(f"\n{'='*80}")
    print(f" 【对比分析】")
    print(f"{'='*80}")
    
    print(f"\n 1. 覆盖公司对比:")
    print(f"    - 普通混合: {list(hybrid_companies.keys())}")
    print(f"    - 两阶段检索: {list(two_stage_companies.keys())}")
    
    hybrid_ids = set([doc_id for doc_id, _, _, _ in hybrid_results])
    two_stage_ids = set([doc_id for doc_id, _, _, _ in two_stage_results])
    
    overlap = hybrid_ids & two_stage_ids
    only_hybrid = hybrid_ids - two_stage_ids
    only_two_stage = two_stage_ids - hybrid_ids
    
    print(f"\n 2. 片段重叠情况:")
    print(f"    - 共同片段: {len(overlap)} 个 ({len(overlap)/len(hybrid_ids)*100:.1f}%)")
    print(f"    - 仅混合检索: {len(only_hybrid)} 个")
    print(f"    - 仅两阶段检索: {len(only_two_stage)} 个")
    
    print(f"\n 3. 关键差异:")
    if len(only_hybrid) > 0:
        print(f"    混合检索独有片段: {list(only_hybrid)[:5]}")
    if len(only_two_stage) > 0:
        print(f"    两阶段检索独有片段: {list(only_two_stage)[:5]}")
    
    return {
        'hybrid': hybrid_results,
        'two_stage': two_stage_results,
        'hybrid_companies': hybrid_companies,
        'two_stage_companies': two_stage_companies
    }




In [12]:
# 运行对比测试
query = '对比5家公司的净利润率（净利润/营业收入），哪家公司盈利能力最强？请按从高到低排序。'
comparison = compare_retrieval_methods(query, top_k=10)


 对比测试：两阶段检索 vs 普通混合检索
 查询问题: 对比5家公司的净利润率（净利润/营业收入），哪家公司盈利能力最强？请按从高到低排序。
 返回数量: 10

────────────────────────────────────────────────────────────────────────────────
 【方法1】普通混合检索 (Hybrid + RRF)
────────────────────────────────────────────────────────────────────────────────

 检索到 10 条片段
 覆盖公司数: 5
 公司分布: {'盛和资源控股股份有限公司': [3009, 3013], '合诚工程咨询集团股份有限公司': [895, 985], '三角轮胎股份有限公司': [375, 372], '河北汇金机电股份有限公司': [2217, 2617], '山东新华锦国际股份有限公司': [1452, 1659]}

 片段详情:
   [1] 盛和资源控股股份有限公司 | ID:3009 | 得分:0.0303
       <html><body><table><tr><td></td><td>第一季度 （1-3月份）</td><td>第二季度 （4-6月份）</td><td>第三季度 （7-9月份）</td><td>第...
   [2] 合诚工程咨询集团股份有限公司 | ID:895 | 得分:0.0164
       <html><body><table><tr><td></td><td>第一季度 （1-3月份）</td><td>第二季度 （4-6月份）</td><td>第三季度 （7-9月份）</td><td>第...
   [3] 三角轮胎股份有限公司 | ID:375 | 得分:0.0164
       <html><body><table><tr><td colspan="4"></td></tr><tr><td>列）</td><td></td><td></td><td></td></tr><tr>...
   [4] 河北汇金机电股份有限公司 | ID:2217 | 得分:0.0161
       <html><body><table><tr><td></td><t

In [11]:
def find_chunks_by_content(search_text: str, show_all: bool = True):
    """
    根据搜索文本在数据库中查找所有匹配的chunk
    
    Args:
        search_text: 搜索文本（如 "销售费用10.29%增长"）
        show_all: 是否显示所有匹配结果
    
    Returns:
        list: [{"chunk_id": doc_id, "source_file": source_file}, ...]
               所有包含search_text的chunk
    """
    from rag.database import load_all_chunks
    
    print(f"\n{'='*80}")
    print(f" 【根据内容查找Chunk】")
    print(f" 搜索文本: {search_text}")
    print('='*80)
    
    chunks_dict = load_all_chunks()
    matches = []
    
    for doc_id, (text_for_bm25, source_file, chunk_text) in chunks_dict.items():
        # 在原始文本或bm25文本中搜索
        if search_text in chunk_text or search_text in text_for_bm25:
            matches.append({
                "chunk_id": doc_id,
                "source_file": source_file,
                "chunk_text": chunk_text
            })
    
    if matches:
        print(f"\n 找到 {len(matches)} 个匹配的chunk:")
        print("-" * 80)
        
        if show_all:
            for i, match in enumerate(matches, 1):
                company = match['source_file'].replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
                print(f"\n[{i}] Chunk ID: {match['chunk_id']}")
                print(f"    来源: {company}")
                print(f"    文件: {match['source_file']}")
                # 显示搜索文本在内容中的位置
                idx = match['chunk_text'].find(search_text)
                if idx != -1:
                    start = max(0, idx - 50)
                    end = min(len(match['chunk_text']), idx + len(search_text) + 50)
                    context = match['chunk_text'][start:end]
                    print(f"    上下文: ...{context}...")
        else:
            # 只显示前3个
            for i, match in enumerate(matches[:3], 1):
                company = match['source_file'].replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
                print(f"\n[{i}] Chunk ID: {match['chunk_id']}")
                print(f"    来源: {company}")
                print(f"    文件: {match['source_file']}")
            
            if len(matches) > 3:
                print(f"\n (还有 {len(matches)-3} 个匹配，设置 show_all=True 可查看全部)")
        
        # 返回简化的结果列表
        return [{"chunk_id": m["chunk_id"], "source_file": m["source_file"]} for m in matches]
    else:
        print(f"\n 未找到包含 '{search_text}' 的chunk")
        return []




In [34]:
find_chunks_by_content(search_text='<html><body><table><tr><td></td><td>2019年</td><td>2018年</td><td>同比增减</td><td>重大变动说明</td></tr><tr><td>销售费用</td><td>66,731,323.61</td><td>97,664,112.32</td><td></td><td>-31.67%')


 【根据内容查找Chunk】
 搜索文本: <html><body><table><tr><td></td><td>2019年</td><td>2018年</td><td>同比增减</td><td>重大变动说明</td></tr><tr><td>销售费用</td><td>66,731,323.61</td><td>97,664,112.32</td><td></td><td>-31.67%

 找到 1 个匹配的chunk:
--------------------------------------------------------------------------------

[1] Chunk ID: 2238
    来源: 河北汇金机电股份有限公司
    文件: 河北汇金机电股份有限公司 2019年度报告.md
    上下文: ...<html><body><table><tr><td></td><td>2019年</td><td>2018年</td><td>同比增减</td><td>重大变动说明</td></tr><tr><td>销售费用</td><td>66,731,323.61</td><td>97,664,112.32</td><td></td><td>-31.67%合并范围变化所致</td></tr><tr><td>管理费用</td><td>47,111,192....


[{'chunk_id': 2238, 'source_file': '河北汇金机电股份有限公司 2019年度报告.md'}]

In [ ]:
def get_chunk_ranking(query: str, chunk_ids, mode: str = "two_stage", top_k: int = 10, use_summary_for_table: bool = True):
    """
    获取指定chunk_id在检索结果中的排名
    
    Args:
        query: 查询问题
        chunk_ids: 要查询的chunk_id，可以是单个整数或列表
        mode: 检索模式 ("two_stage" 或 "hybrid")
        top_k: 检索返回数量
        use_summary_for_table: 如果chunk是表格，用summary单独计算排名
    
    Returns:
        dict: 每个chunk_id的排名信息
    """
    from rag.database import load_all_chunks, get_summary_dict, is_table_chunk
    
    # 支持单个整数或列表
    if isinstance(chunk_ids, int):
        chunk_ids = [chunk_ids]
    
    print(f"\n{'='*80}")
    print(f" 【Chunk排名查询】")
    print(f" 查询问题: {query}")
    print(f" 目标Chunk IDs: {chunk_ids}")
    print(f" 检索模式: {mode} | 返回数量: {top_k}")
    print(f" 使用summary计算表格排名: {use_summary_for_table}")
    print('='*80)
    
    # 执行检索
    if mode == "two_stage":
        results = two_stage_retrieval(query, top_k=top_k)
    elif mode == "hybrid":
        results = hybrid_search(query, top_k=top_k)
    else:
        raise ValueError(f"不支持的检索模式: {mode}")
    
    # 建立doc_id到排名的映射
    id_to_rank = {}
    for rank, (doc_id, chunk_text, score, source_file) in enumerate(results, 1):
        id_to_rank[doc_id] = {
            'rank': rank,
            'score': score,
            'source_file': source_file
        }
    
    # 获取summary字典
    summary_dict = get_summary_dict()
    chunks_dict = load_all_chunks()
    
    # 检查每个目标chunk_id的排名
    print(f"\n 排名结果:")
    print("-" * 80)
    
    ranking_results = {}
    for chunk_id in chunk_ids:
        # 检查是否是表格
        is_table = chunk_id in summary_dict
        
        # 先检查在主检索中的排名
        if chunk_id in id_to_rank:
            info = id_to_rank[chunk_id]
            company = info['source_file'].replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
            table_marker = " [表格]" if is_table else ""
            
            print(f"\n Chunk ID {chunk_id}{table_marker}:")
            print(f"   来源: {company}")
            print(f"   [{mode}检索] 排名第 {info['rank']} | 得分: {info['score']:.4f}")
            
            ranking_results[chunk_id] = {
                'found': True,
                'rank': info['rank'],
                'score': info['score'],
                'source_file': info['source_file'],
                'is_table': is_table
            }
            
            # 如果是表格，额外显示summary单独检索的排名
            if is_table and use_summary_for_table:
                # 用summary进行向量检索
                from rag import dense_search
                summary_results = dense_search(query, top_k=500)
                summary_rank = None
                for rank, (doc_id, _, score, _) in enumerate(summary_results, 1):
                    if doc_id == chunk_id:
                        summary_rank = rank
                        break
                
                if summary_rank:
                    print(f"   [向量检索-Summary] 排名第 {summary_rank}")
                else:
                    print(f"   [向量检索-Summary] 未进top-500")
                ranking_results[chunk_id]['summary_rank'] = summary_rank
                
        else:
            # 主检索中未找到
            table_marker = " [表格]" if is_table else ""
            print(f"\n Chunk ID {chunk_id}{table_marker}:")
            print(f"   [{mode}检索] 未在前 {top_k} 名中 ❌")
            
            ranking_results[chunk_id] = {
                'found': False,
                'rank': None,
                'score': None,
                'is_table': is_table
            }
            
            # 如果是表格，显示summary单独检索的排名
            if is_table and use_summary_for_table:
                # 用summary进行向量检索
                from rag import dense_search
                summary_results = dense_search(query, top_k=500)
                summary_rank = None
                for rank, (doc_id, _, score, _) in enumerate(summary_results, 1):
                    if doc_id == chunk_id:
                        summary_rank = rank
                        break
                
                if summary_rank:
                    print(f"   [向量检索-Summary] 排名第 {summary_rank}")
                    ranking_results[chunk_id]['summary_rank'] = summary_rank
                else:
                    print(f"   [向量检索-Summary] 未进top-500")
                    ranking_results[chunk_id]['summary_rank'] = None
            else:
                ranking_results[chunk_id]['summary_rank'] = None
            
            print(f"   ⚠️ 建议: 增加top_k值或检查summary质量")
    
    # 显示检索结果的top-N作为参考
    print(f"\n{'='*80}")
    print(f" 检索结果 Top-5 (参考)")
    print('='*80)
    for rank, (doc_id, chunk_text, score, source_file) in enumerate(results[:5], 1):
        company = source_file.replace('2019 年年度报告.md', '').replace(' 2019年度报告.md', '').replace('.md', '')
        is_table = doc_id in summary_dict
        table_marker = " [表格]" if is_table else ""
        marker = " <<< 目标" if doc_id in chunk_ids else ""
        print(f"\n[{rank}] ID:{doc_id} {company}{table_marker} {marker}")
        print(f"    得分: {score:.4f}")
        print(f"    预览: {chunk_text[:100]}...")
    
    return ranking_results




In [39]:
# 测试：用summary计算排名
result = get_chunk_ranking(
    query="新华锦销售费用",
    chunk_ids=[1456],
    mode="hybrid",
    top_k=100,
    use_summary_for_table=True
)


 【Chunk排名查询】
 查询问题: 新华锦销售费用
 目标Chunk IDs: [1456]
 检索模式: hybrid | 返回数量: 100
 使用summary计算表格排名: True

 排名结果:
--------------------------------------------------------------------------------

 Chunk ID 1456 [表格]: 未在前 100 名中 ❌
   └─ Summary单独检索排名: 339
   ⚠️ 可能需要增加 top_k 值

 检索结果 Top-5 (参考)

[1] ID:1674 山东新华锦国际股份有限公司 [表格] 
    得分: 0.0323
    预览: <html><body><table><tr><td>公司名称</td><td>结汇金额(美元)</td><td>收益金额</td></tr><tr><td>新华锦集团山东海诚进出口有限公司</td>...

[2] ID:1669 山东新华锦国际股份有限公司 [表格] 
    得分: 0.0301
    预览: <html><body><table><tr><td>关联方</td><td>关联交易内容</td><td>本期发生额</td><td>上期发生额</td></tr><tr><td>新华锦集公山东海锦...

[3] ID:1184 山东新华锦国际股份有限公司 
    得分: 0.0164
    预览: 公司代码：600735  

公司简称：新华锦  

![](images/4a33585b87c8502c5951971cc89facf4826194130fece77dc4f6c39424359c...

[4] ID:1686 山东新华锦国际股份有限公司 [表格] 
    得分: 0.0164
    预览: <html><body><table><tr><td>被投资单位</td><td>期初余额</td><td>本期增加</td><td>本期减 少</td><td>期末余额</td><td>本期计 提减...

[5] ID:1448 山东新华锦国际股份有限公司 [表格] 
    得分: 0.0159
    预览: <html><body><table><tr